# Test HDF5 performance

This notebook is for testing the performance of swapping and loading temporals with TemporalManager and HDF5 files.

We will create a project with M branches and N buildings per branch. Each building will have a large number of temporals. We will then test the performance of swapping and loading temporals by taking a round trip:
- Temporals will initially be in mode "loaded"
- Test access time, pickle, hdf5 file and memory sizes
- Swap out all temporals to HDF5 files
- Test access time, pickle, hdf5 file and memory sizes
- Set all temporals to "lazy" mode (which won't change anything)
- Test access time, pickle, hdf5 file and memory sizes – this will implicitly load all temporals

In [ ]:
import logging
import os
import time
import pandas as pd
from pathlib import Path
import numpy as np
import h5py
from pympler import asizeof
from typing import Tuple
import tempfile

from odeon.model import Branch, Project, HeatDemand
from odeon.io import FileAdapter, TemporalManager
from odeon.processing.utils import file_size_as_str, bytes_as_str

logging.getLogger("odeon.io.file_adapter").disabled = True

In [ ]:
M = 5
N = 100

In [ ]:
tempdir = tempfile.TemporaryDirectory()
print(f"Using temporary directory: {tempdir.name}")

# create project with branches and heat demands
project = Project(
    name="Test project",
    file_adapter=FileAdapter(dir=Path(tempdir.name)/"test_hdf_performance_data"),
    temporal_manager=TemporalManager(),
)

for i in range(M):
    branch = Branch(year=2025, project=project)
    for j in range(N // 2):  # two temporals per heat demand
        heat_demand = HeatDemand()
        heat_demand.input_flow = pd.Series(np.random.uniform(low=0.0, high=10.0, size=8760)) # temporal no. 1
        heat_demand.set_factor(pd.Series(np.random.uniform(low=0.0, high=10.0, size=8760))) # temporal no. 2
        branch.add_objects(heat_demand)

# store to pickle:
pickle_path = project.file_adapter.write_to_dir()

In [ ]:
n_temporals = sum([len(b.temporals_recursive()) for b in project.branches])
print(f"Project with {M} branches and {n_temporals} temporals stored to {pickle_path}")

## Performance and memory profiling

In [ ]:
def get_all_hdf_files_in_dir(dir: Path) -> list[Path]:
    return [p for p in dir.iterdir() if p.suffix == ".hdf"]

def get_size_of_all_hdf_files_in_dir(dir: Path) -> int:
    hdf_files = get_all_hdf_files_in_dir(dir)
    return sum([os.path.getsize(p) for p in hdf_files])

def memory_size_of_project(project: Project) -> int:
    return asizeof.asizeof(project)

def print_sizes(pickle_path: Path, project: Project):
    pickle_size = file_size_as_str(pickle_path)
    hdf_size = bytes_as_str(get_size_of_all_hdf_files_in_dir(project.file_adapter.dir))
    memory_size = bytes_as_str(memory_size_of_project(project))

    print(f"Memory size of project: {memory_size}")
    print(f"Pickle size: {pickle_size}")
    print(f"HDF size: {hdf_size} ({len(get_all_hdf_files_in_dir(project.file_adapter.dir))} files)")

def measure_access_time(project: Project) -> Tuple[float, int]:
    start_time = time.time()
    n = 0
    for branch in project.branches:
        for temporal in branch.temporals_recursive():
            n += 1
            _ = temporal.series.sum()
    end_time = time.time()
    elapsed_time = end_time - start_time
    return n, elapsed_time

def print_access_time(project: Project):
    n, elapsed_time = measure_access_time(project)
    print(f"Access time for all temporals: {elapsed_time:.2f} s ({elapsed_time / n * 1000:.2f} ms per temporal)")

def write_files_and_print_time(project: Project):
    tic = time.time()
    project.file_adapter.write_to_dir() # rewrite pickle
    tac = time.time()
    project.file_adapter.flush_hdf_files()
    toc = time.time()
    print(f"Rewrote pickle with swapped temporals in {tac - tic:.2f} s, flushed HDF files in {toc - tac:.2f} s")

### loaded

In [ ]:
print_sizes(pickle_path, project)
print_access_time(project)

### swap operation

In [ ]:
tic = time.time()
project.temporal_manager.swap_temporals()
toc = time.time()
print(f"Swapped all temporals in {toc - tic:.2f} s ({(toc - tic) / n_temporals * 1000:.2f} ms per temporal)")

In [ ]:
write_files_and_print_time(project)

### swapped

In [ ]:
print_sizes(pickle_path, project)
print_access_time(project)

### lazy

In [ ]:
# set all temporals to lazy - won't change anything:
project.temporal_manager.set_swap_mode("lazy")

print_access_time(project) # will load all temporals into memory again

In [ ]:
# after accessing once, all temporals are in memory
project.file_adapter.clean(delete_files=True)
print_sizes(pickle_path, project)

### Swap or load based on access times

In [ ]:
# get all timeseries of all branches and store them in a dictionary per branch with the id as key:
all_series = {
    branch.id: {temporal.id: temporal for temporal in branch.temporals_recursive()}
    for branch in project.branches
}

In [ ]:
import matplotlib.pyplot as plt

def plot_access_times():

    access_times = [
        temporal.n_accesses for branch in all_series.values() for temporal in branch.values()
    ]

    plt.hist(access_times, bins=20)
    plt.xlabel("Access Time")
    plt.ylabel("Frequency")
    plt.title("Access Times Histogram")
    plt.show()

def plot_access_times_by_swap_state():
    access_times_swapped = [
        temporal.n_accesses for branch in all_series.values() for temporal in branch.values() if temporal._swapped
    ]
    access_times_loaded = [
        temporal.n_accesses for branch in all_series.values() for temporal in branch.values() if not temporal._swapped
    ]

    plt.hist(access_times_swapped, bins=20, label="Swapped")
    plt.hist(access_times_loaded, bins=20, label="Loaded")
    plt.xlabel("Access Time")
    plt.ylabel("Frequency")
    plt.title("Access Times Histogram by Swap State")
    plt.legend()
    plt.show()    

In [ ]:
# randomly access some temporals to change their access times:
import random
for _ in range(1000):
    branch = random.choice(list(all_series.values()))
    temporal = random.choice(list(branch.values()))
    _ = temporal.series.sum()  # Access the series to update access time

In [ ]:
plot_access_times()

In [ ]:
project.temporal_manager.swap_or_load_temporals(threshold_swap=8, threshold_load=10)

In [ ]:
plot_access_times_by_swap_state()

## Cleanup

In [ ]:
project.file_adapter.close_hdf_files()

# clear files from directory:
if project.file_adapter.dir.exists():
    for file in os.listdir(project.file_adapter.dir):
        os.remove(project.file_adapter.dir / file)